In [25]:
import filtering
import parcels
import numpy as np
from datetime import timedelta
from glob import glob
import matplotlib.pyplot as plt
import xarray as xr
import os

def u2rho_2d (var_u):
    [Mp,L]=var_u.shape
    Lp=L+1
    Lm=L-1
    var_rho=np.zeros((Mp,Lp))
    var_rho[:,1:-1]=0.5*(var_u[:,1:]+var_u[:,:-1])
    var_rho[:,0]=var_rho[:,1]
    var_rho[:,-1]=var_rho[:,-2]
    return var_rho
    
def v2rho_2d (var_v):
    [M,Lp]=var_v.shape
    Mp=M+1
    Mm=M-1
    var_rho=np.zeros((Mp,Lp))
    var_rho[1:-1,:]=0.5*(var_v[1:,:]+var_v[:-1,:])
    var_rho[0,:]=var_rho[1,:]
    var_rho[-1,:]=var_rho[-2,:]
    return var_rho

def u2rho_3d (var_u):
    [N,Mp,L]=var_u.shape
    Lp=L+1
    Lm=L-1
    var_rho=np.zeros((N,Mp,Lp))
    var_rho[:,:,1:-1]=0.5*(var_u[:,:,1:]+var_u[:,:,:-1])
    var_rho[:,:,0]=var_rho[:,:,1]
    var_rho[:,:,-1]=var_rho[:,:,-2]
    return var_rho
    
def v2rho_3d (var_v):
    [N,M,Lp]=var_v.shape
    Mp=M+1
    Mm=M-1
    var_rho=np.zeros((N,Mp,Lp))
    var_rho[:,1:-1,:]=0.5*(var_v[:,1:,:]+var_v[:,:-1,:])
    var_rho[:,0,:]=var_rho[:,1,:]
    var_rho[:,-1,:]=var_rho[:,-2,:]
    return var_rho

def spheric_dist(lat1, lat2, lon1, lon2):
    """
    Compute the spherical distance between two points on Earth.
    
    Parameters:
    lat1, lat2 : array-like
        Latitude of the two points (in degrees).
    lon1, lon2 : array-like
        Longitude of the two points (in degrees).
    
    Returns:
    dist : array-like
        The spherical distance between the points (in meters).
    """
    
    # Earth radius in meters
    R = 6367442.76
    
    # Determine proper longitudinal shift
    l = np.abs(lon2 - lon1)
    l[l >= 180] = 360 - l[l >= 180]
    
    # Convert decimal degrees to radians
    deg2rad = np.pi / 180
    lat1 = lat1 * deg2rad
    lat2 = lat2 * deg2rad
    l = l * deg2rad
    
    # Compute the distances
    dist = R * np.arcsin(np.sqrt(((np.sin(l) * np.cos(lat2)) ** 2) + 
                                 ((np.sin(lat2) * np.cos(lat1)) - 
                                  (np.sin(lat1) * np.cos(lat2) * np.cos(l))) ** 2))
    
    return dist

def spheric_dist_one(lat1, lat2, lon1, lon2):
    """计算两个经纬度点之间的球面距离（标量版）"""
    # 处理经度差
    l = np.abs(lon2 - lon1)
    if l >= 180:
        l = 360 - l
    
    # 转换为弧度
    deg2rad = np.pi / 180
    lat1_rad = lat1 * deg2rad
    lat2_rad = lat2 * deg2rad
    l_rad = l * deg2rad
    
    # 球面距离公式
    distance = 6371 * np.arccos(
        np.sin(lat1_rad) * np.sin(lat2_rad) + 
        np.cos(lat1_rad) * np.cos(lat2_rad) * np.cos(l_rad)
    )
    return distance
def transunit_spher2flat(u,v,lat):
    v1=v*1852*60
    u1=u*1852*60*np.cos(lat*np.pi/180)
    return u1,v1

def trans_vel_roms(u,v,angle):
    # np.cos(angle*np.pi/180)
    # np.sin(angle*np.pi/180)
    u_east=u*np.cos(angle*np.pi/180)-v*np.sin(angle*np.pi/180)
    v_north=u*np.sin(angle*np.pi/180)+v*np.cos(angle*np.pi/180)
    return u_east,v_north

from scipy.interpolate import griddata

def griddata_matlab(lona,lata,ugos,lon_rho,lat_rho):
    points = np.column_stack((lona.ravel(), lata.ravel()))
    values = ugos.ravel()
    
    # 步骤2: 创建目标网格点
    target_points = np.column_stack((lon_rho.ravel(), lat_rho.ravel()))
    
    # 步骤3: 执行插值
    ugos_interp = griddata(points, values, target_points, method='linear')
    
    # 步骤4: 重塑为原始网格形状
    ugos_interp = ugos_interp.reshape(lon_rho.shape)
    return ugos_interp

def snake_scan(matrix):
    """
    ravel() for particles
    matrix: 
        
    flat_array: 1d arrays
    """
    rows, cols = matrix.shape
    result = []
    
    for i in range(rows):
        if i % 2 == 0:  # 偶数行：从左到右
            result.extend(matrix[i, :])
        else:           # 奇数行：从右到左
            result.extend(matrix[i, ::-1])
    
    return np.array(result)

import numpy as np
from scipy.interpolate import griddata

def particle_roughdistr_onetime(lonp, latp, cI, cJ, NP, Range, shape='circle'):
    '''    
    para:
        lonp, latp: grid
        cI, cJ: center of region
        NP: numbers of p
        Range: sub region
        shape:  ('box' or 'circle')
    '''
    squareP = int(np.sqrt(NP))
    Len = int(Range)
    
    # 
    if Len % 2 == 0:
        II = [int(cI - Len/2), int(cI + Len/2)]
        JJ = [int(cJ - Len/2), int(cJ + Len/2)]
    else:
        II = [int(cI - (Len+1)/2), int(cI + Len/2)]
        JJ = [int(cJ - (Len+1)/2), int(cJ + Len/2)]
    
    # sub region
    lonp1 = lonp[JJ[0]:JJ[1], II[0]:II[1]]
    latp1 = latp[JJ[0]:JJ[1], II[0]:II[1]]
    
    # 
    Iin, Jin = np.meshgrid(np.arange(lonp1.shape[0]), 
                          np.arange(lonp1.shape[1]))
    
    points = np.column_stack((Iin.ravel(), Jin.ravel()))

    # choose shape
    if shape == 'box':
        # 
        I1 = np.linspace(0, lonp1.shape[0]-1, squareP)
        J1 = np.linspace(0, lonp1.shape[1]-1, squareP)
        I11, J11 = np.meshgrid(I1, J1)
        
        # 
        lonp2 = griddata(points, lonp1.ravel(), (I11, J11), method='linear')
        latp2 = griddata(points, latp1.ravel(), (I11, J11), method='linear')
        return snake_scan(lonp2), snake_scan(latp2)
    
    elif shape == 'circle':

        center_x = lonp1.shape[0] / 2
        center_y = lonp1.shape[1] / 2
        
        radius = min(lonp1.shape[0], lonp1.shape[1]) / 2
        
        r = radius * np.sqrt(np.random.rand(NP))  #
        theta = 2 * np.pi * np.random.rand(NP)
        
        # 
        x = center_x + r * np.cos(theta)
        y = center_y + r * np.sin(theta)
        
        # 
        points_target = np.column_stack((x, y))
        lonp2 = griddata(points, lonp1.ravel(), points_target, method='linear')
        latp2 = griddata(points, latp1.ravel(), points_target, method='linear')
        
        return lonp2, latp2



def generate_release_times_hourly(total_particles, particles_per_hour, start_time=0):
    """
    :
        total_particles: 
        particles_per_hour: 
        start_time: 
    :
        timep2: 
    """
    total_particles = int(total_particles)
    particles_per_hour = int(particles_per_hour)

    hours_needed = int(np.ceil(total_particles / particles_per_hour))

    timep2 = []
    for hour in range(hours_needed):
        particles_this_hour = min(particles_per_hour, total_particles - len(timep2))
        hour_start = start_time + hour * 3600
        timep2.extend([hour_start] * particles_this_hour)
    
    return np.array(timep2)

In [26]:
grid_dir='/meddy/simingzhang/Data/RB_iceland_data/'
wave_dir='/meddy/simingzhang/Data/Parcels_data/'
# nowave_dir='/meddy/simingzhang/Data/RB_iceland_data/iceland_no_wave/'
grdname='niskin2km_500m_grd.nc'
# hisname_w='z_niskin2km_his_hf_depth_500m_grd.0002.nc'
# hisname_nw='z_niskin2km_his_smooth_depth_500m_grd.0002.nc'

grdname=f'{grid_dir}{grdname}'
# wavename=f'{wave_dir}{hisname_w}'
# nowavename=f'{nowave_dir}{hisname_nw}'

# wavename=f'{wave_dir}wavecase_modified_cg.nc'
# nowavename=f'{wave_dir}nowavecase_modified_cg.nc'
wavename=f'{wave_dir}test2_wave_fix.nc'


In [27]:
grid=xr.open_dataset(grdname)
grid = grid.swap_dims({'eta_u': 'eta_rho','xi_v':'xi_rho'})

lon_rho=grid['lon_rho']
lat_rho=grid['lat_rho']
h=grid['h']
f=np.nanmean(grid['f'].values)
angle=grid['angle'].values
lonmin=np.min(lon_rho.values)
lonmax=np.max(lon_rho.values)
latmin=np.min(lat_rho.values)
latmax=np.max(lat_rho.values)
lonmin,lonmax,latmin,latmax
grid

<xarray.Dataset> Size: 19MB
Dimensions:    (one: 1, eta_rho: 287, xi_rho: 287, bath: 1, xi_u: 286,
                eta_v: 286, eta_psi: 286, xi_psi: 286)
Dimensions without coordinates: one, eta_rho, xi_rho, bath, xi_u, eta_v,
                                eta_psi, xi_psi
Data variables: (12/34)
    xl         (one) float64 8B ...
    el         (one) float64 8B ...
    depthmin   (one) float64 8B ...
    depthmax   (one) float64 8B ...
    spherical  (one) |S1 1B ...
    angle      (eta_rho, xi_rho) float64 659kB -10.14 -10.14 ... -11.33 -11.33
    ...         ...
    lat_v      (eta_v, xi_rho) float64 657kB ...
    lat_psi    (eta_psi, xi_psi) float64 654kB ...
    mask_rho   (eta_rho, xi_rho) float64 659kB ...
    mask_u     (eta_rho, xi_u) float64 657kB ...
    mask_v     (eta_v, xi_rho) float64 657kB ...
    mask_psi   (eta_psi, xi_psi) float64 654kB ...
Attributes:
    title:    Solomon Model
    date:     08-Apr-2018
    type:     ROMS grid file

# set dict

In [28]:
# 创建变量名列表
variable_names = ["U", "V"]


# 创建字典
filenames = {var: wavename for var in variable_names}

# 打印结果
# print(filenames)
filenames

{'U': '/meddy/simingzhang/Data/Parcels_data/test2_wave_fix.nc',
 'V': '/meddy/simingzhang/Data/Parcels_data/test2_wave_fix.nc'}

In [29]:
all_vars = ["U", "V"]

variables = {
    var: ("u_rho" if var == "U" else "v_rho")
    for var in all_vars
}

# 维度模板
dim_template = {
    "lat": "lat_rho",
    "lon": "lon_rho",
    "time": "time",
    "depth": "depth"
}

# 创建 dimensions 字典
dimensions = {var: dim_template.copy() for var in all_vars}
indices = {"depth": [0]}
dimensions
# fieldset = parcels.FieldSet.from_netcdf(filenames, variables, dimensions,allow_time_extrapolation=True)


{'U': {'lat': 'lat_rho', 'lon': 'lon_rho', 'time': 'time', 'depth': 'depth'},
 'V': {'lat': 'lat_rho', 'lon': 'lon_rho', 'time': 'time', 'depth': 'depth'}}

In [32]:
# f = filtering.LagrangeFilter(
# 	f"{wave_dir}waves", filenames, variables, dimensions,
# 	sample_variables=["U","V"], mesh="spherical",
# 	window_size=timedelta(days=2).total_seconds(),
#     indices=indices,
#     # runtime=timedelta(hours=120),
#     # dt=timedelta(seconds=600),
# )

ff = filtering.LagrangeFilter(
	f"{wave_dir}waves", filenames, variables, dimensions,
	sample_variables=["U","V"], mesh="spherical",
	window_size=timedelta(days=2).total_seconds(),
    highpass_frequency=1/(14*3600)
)

# periodic domain
# ff.make_zonally_periodic()
# ff.make_meridionally_periodic()

ff.advection_dt=600
t1=timedelta(days=2).total_seconds()
ff.filter(times=[t1])

INFO: Compiled SamplingParticlesample_kernel ==> /tmp/parcels-979991/54974d4e5e35d3c097e2aceedc9efb5a_0.so
INFO:parcels.tools.loggers:Compiled SamplingParticlesample_kernel ==> /tmp/parcels-979991/54974d4e5e35d3c097e2aceedc9efb5a_0.so
INFO: Compiled SamplingParticleAdvectionRK4sample_kernel ==> /tmp/parcels-979991/48a98cca4bdaa940323490ce81144025_0.so
INFO:parcels.tools.loggers:Compiled SamplingParticleAdvectionRK4sample_kernel ==> /tmp/parcels-979991/48a98cca4bdaa940323490ce81144025_0.so
/home/simingzhang/anaconda3/envs/parcels/lib/python3.11/site-packages/parcels/field.py:1325: RuntimeWarning: invalid value encountered in cast
  data = lib.concatenate([data_to_concat, data[tindex+1:, :]], axis=0)
/home/simingzhang/anaconda3/envs/parcels/lib/python3.11/site-packages/parcels/field.py:1327: RuntimeWarning: invalid value encountered in cast
  data = lib.concatenate([data[:tindex, :], data_to_concat], axis=0)
/home/simingzhang/anaconda3/envs/parcels/lib/python3.11/site-packages/parcels/fi

/home/simingzhang/anaconda3/envs/parcels/lib/python3.11/site-packages/parcels/field.py:1325: RuntimeWarning: invalid value encountered in cast
  data = lib.concatenate([data_to_concat, data[tindex+1:, :]], axis=0)
/home/simingzhang/anaconda3/envs/parcels/lib/python3.11/site-packages/parcels/field.py:1327: RuntimeWarning: invalid value encountered in cast
  data = lib.concatenate([data[:tindex, :], data_to_concat], axis=0)
/home/simingzhang/anaconda3/envs/parcels/lib/python3.11/site-packages/parcels/field.py:1327: RuntimeWarning: invalid value encountered in cast
  data = lib.concatenate([data[:tindex, :], data_to_concat], axis=0)
/home/simingzhang/anaconda3/envs/parcels/lib/python3.11/site-packages/parcels/field.py:1327: RuntimeWarning: invalid value encountered in cast
  data = lib.concatenate([data[:tindex, :], data_to_concat], axis=0)
/home/simingzhang/anaconda3/envs/parcels/lib/python3.11/site-packages/parcels/field.py:1327: RuntimeWarning: invalid value encountered in cast
  data 

# you should fixed time first

In [48]:
import xarray as xr
import numpy as np

def diagnose_dataset(filename):
    """诊断数据集结构"""
    print("=== 数据集诊断 ===")
    
    try:
        ds = xr.open_dataset(filename)
        print(f"✅ 成功打开文件: {filename}")
        
        print("\n=== 维度信息 ===")
        for dim_name, dim_size in ds.dims.items():
            print(f"维度 '{dim_name}': 大小={dim_size}")
            dim_var = ds[dim_name]
            
            # 检查属性
            if hasattr(dim_var, 'ncattrs'):
                attrs = dim_var.ncattrs()
                print(f"  属性: {attrs}")
                if 'units' in attrs:
                    print(f"  单位: {dim_var.units}")
                if 'calendar' in attrs:
                    print(f"  日历: {dim_var.calendar}")
        
        print("\n=== 变量信息 ===")
        for var_name in ds.variables:
            if var_name not in ds.dims:  # 跳过维度变量
                var = ds[var_name]
                print(f"变量 '{var_name}': 形状={var.shape}, 维度={var.dims}")
        
        print("\n=== 时间相关维度 ===")
        time_candidates = []
        for dim in ds.dims:
            dim_var = ds[dim]
            if hasattr(dim_var, 'units') and 'time' in str(dim_var.units).lower():
                time_candidates.append(dim)
                print(f"✅ 时间维度候选: {dim} (单位: {dim_var.units})")
            elif 'time' in dim.lower():
                time_candidates.append(dim)
                print(f"⚠️  可能的时间维度: {dim}")
        
        ds.close()
        return time_candidates
        
    except Exception as e:
        print(f"❌ 无法打开文件 {filename}: {e}")
        return []

# 运行诊断
filename = wavename  # 替换为实际文件路径
time_dims = diagnose_dataset(filename)

if time_dims:
    print(f"\n🎯 建议使用的时间维度: {time_dims[0]}")
else:
    print("\n❌ 未找到时间维度")

=== 数据集诊断 ===
✅ 成功打开文件: /meddy/simingzhang/Data/Parcels_data/wavecase_modified_vel_cg_kaiser_corr.nc

=== 维度信息 ===
维度 'time': 大小=2148
维度 'depth': 大小=1
维度 'eta_rho': 大小=287
维度 'xi_rho': 大小=287
维度 'xi_u': 大小=286
维度 'eta_v': 大小=286
维度 'depth_2': 大小=1

=== 变量信息 ===
变量 'Ro': 形状=(2148, 1, 287, 287), 维度=('time', 'depth', 'eta_rho', 'xi_rho')
变量 'divof': 形状=(2148, 1, 287, 287), 维度=('time', 'depth', 'eta_rho', 'xi_rho')
变量 'ocean_time': 形状=(2148,), 维度=('time',)
变量 'u': 形状=(2148, 1, 287, 286), 维度=('time', 'depth', 'eta_rho', 'xi_u')
变量 'u_rho': 形状=(2148, 1, 287, 287), 维度=('time', 'depth', 'eta_rho', 'xi_rho')
变量 'v': 形状=(2148, 1, 286, 287), 维度=('time', 'depth', 'eta_v', 'xi_rho')
变量 'v_rho': 形状=(2148, 1, 287, 287), 维度=('time', 'depth', 'eta_rho', 'xi_rho')
变量 'w': 形状=(2148, 1, 287, 287), 维度=('time', 'depth', 'eta_rho', 'xi_rho')
变量 'Th1': 形状=(2148, 1, 287, 287), 维度=('time', 'depth_2', 'eta_rho', 'xi_rho')
变量 'Th2': 形状=(2148, 1, 287, 287), 维度=('time', 'depth_2', 'eta_rho', 'xi_rho')
变量 'Th3': 形状=

/tmp/ipykernel_2705189/2019567551.py:13: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  for dim_name, dim_size in ds.dims.items():


In [46]:
import filtering
import inspect

# 方法1：查看模块位置
print("filtering 模块位置:", filtering.__file__)

filtering 模块位置: /meddy/simingzhang/Analysis/python/python3/lagrangian-filtering/filtering/__init__.py


In [6]:
import xarray as xr
import numpy as np
import os

def fix_time_data(input_file, output_file):
    """修复时间数据"""
    print("=== 修复时间数据 ===")
    
    # 打开原始文件
    ds = xr.open_dataset(input_file)
    
    print(f"原始时间数据: {ds.time.values[:10]}...")  # 显示前10个值
    print(f"时间形状: {ds.time.shape}")
    
    # 检查时间是否全为0
    if np.all(ds.time.values == 0):
        print("❌ 检测到时间数据全为0，需要修复")
        
        # 创建新的时间序列
        # 假设时间间隔为1小时（3600秒），根据实际情况调整
        time_interval = 3600  # 秒
        new_time = np.arange(len(ds.time)) * time_interval
        
        # 更新数据集
        ds = ds.assign_coords(time=new_time)
        
        # 更新时间属性
        ds.time.attrs.update({
            'units': 'seconds since 2020-01-01 00:00:00',  # 根据实际情况调整
            'calendar': 'proleptic_gregorian'
        })
        
        print(f"✅ 新时间数据: {ds.time.values[:10]}...")
        
        # 保存修复后的文件
        ds.to_netcdf(output_file)
        print(f"✅ 已保存修复文件: {output_file}")
        
    else:
        print("✅ 时间数据正常")
        # 如果正常，直接复制文件
        ds.to_netcdf(output_file)
    
    ds.close()
    return output_file

# 修复数据
original_file = "/meddy/simingzhang/Data/Parcels_data/wavecase_modified_vel_cg_kaiser_corr.nc"
fixed_file = "/meddy/simingzhang/Data/Parcels_data/wavecase_modified_vel_cg_kaiser_corr_FIXED.nc"

fixed_file = fix_time_data(original_file, fixed_file)

# 现在使用修复后的文件进行滤波
filenames = fixed_file

=== 修复时间数据 ===
原始时间数据: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]...
时间形状: (2148,)
❌ 检测到时间数据全为0，需要修复
✅ 新时间数据: [    0  3600  7200 10800 14400 18000 21600 25200 28800 32400]...
✅ 已保存修复文件: /meddy/simingzhang/Data/Parcels_data/wavecase_modified_vel_cg_kaiser_corr_FIXED.nc


# test